# Build an LLM from Scratch — Research-Based Assessment (Experiment Notebook)

This notebook extends the class GPT-from-scratch code and runs **every experiment in the assignment**, saving all losses, plots, tables and text samples needed for the final Overleaf report.

| Section | What it produces |
|---|---|
| 0. Setup & config | All settings in **one cell** |
| 1. Dataset | Project Gutenberg book(s), cleaned, split 90/10, dataset statistics |
| 2. Model | Same GPT as in class, plus switches for LayerNorm / shortcut / feed-forward |
| 3. Training engine | Trains, logs train/val loss every epoch, gradient norms, text samples, saves a JSON per run (runs are **cached**, so a Colab disconnect never loses finished work) |
| 4. Baseline | Baseline loss curves + samples |
| 5. Exp 1 | Epochs × learning rate (1e-4, 1e-3, 1e-2; checkpoints at 5…500 epochs) |
| 6. Exp 2 | Transformer layers 1, 3, 5, 7, 12 |
| 7. Exp 3 | Attention heads 1, 2, 4, 8 |
| 8. Exp 4 | Ablations: no LayerNorm, no shortcut, no feed-forward |
| 9. Export | `summary_all.csv`, LaTeX tables, all figures (PDF + PNG), `samples.md`, one zip |

**How to run:** Runtime → Change runtime type → **T4 GPU**. Set `QUICK_TEST = True` first and *Run all* (≈2–3 min) to check everything works, then set it to `False` and *Run all* again for the real results. With `USE_DRIVE = True`, results go to Google Drive and finished runs are skipped if you re-run after a disconnect.

## 0. Setup and configuration

In [ ]:
!pip install -q tiktoken

import os, json, time, math, hashlib, random, re, zipfile, platform, shutil, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tiktoken
import matplotlib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Device:", device, "|", GPU_NAME, "| PyTorch", torch.__version__)

In [ ]:
# ============================ CONFIGURATION ============================
QUICK_TEST = False          # True = tiny smoke test of the whole pipeline (~2-3 min)
USE_DRIVE  = True           # save results to Google Drive (survives disconnects)
RESULTS_DIR_NAME = "llm_scratch_results"
SEED = 123

# ---- Dataset (Project Gutenberg) ----
# 43 = "The Strange Case of Dr Jekyll and Mr Hyde" (~36K GPT-2 tokens, keeps 500-epoch runs affordable).
# Other options: 11 Alice in Wonderland, 46 A Christmas Carol, 1661 Sherlock Holmes, 1342 Pride and Prejudice.
# Put several IDs in the list to train on multiple books combined.
BOOK_IDS = [43]
LOCAL_BOOK_FILES = []       # alternatively, uploaded .txt files, e.g. ["my_book.txt"]
MAX_CHARS = None            # optionally truncate the corpus (e.g. 300_000) to save time
TRAIN_RATIO = 0.90

# ---- Base model: a scaled-down GPT-2 so all ~25 runs fit in free Colab ----
# (the 124M config has 12 layers / 768 dims and is far too slow for 500-epoch sweeps)
BASE_MODEL_CFG = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 256,         # divisible by 1, 2, 4, 8 heads
    "n_heads": 4,
    "n_layers": 6,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "use_norm": True,       # LayerNorm inside every transformer block
    "use_shortcut": True,   # residual connections
    "use_ffn": True,        # feed-forward sub-layer
    "use_final_norm": True, # LayerNorm before the output head (outside the blocks)
}

# ---- Base training settings (used by Baseline, Exp 2, 3, 4) ----
BASE_TRAIN_CFG = {
    "lr": 4e-4,             # same as the class notebook
    "weight_decay": 0.1,
    "batch_size": 8,
    "num_epochs": 30,
    "grad_clip": None,      # None = no clipping (so instabilities stay visible)
}
USE_AMP = False             # True = fp16 mixed precision (~2-3x faster on T4, slightly less exact)

# ---- Experiment grids ----
EXP1_LRS = [1e-4, 1e-3, 1e-2]
EXP1_EPOCH_CHECKPOINTS = [5, 10, 20, 50, 100, 200, 500]
EXP1_INDEPENDENT_RUNS = False   # False: one long run per LR, losses/samples recorded at each checkpoint
                                #        (identical to separate runs, since LR is constant)
                                # True : a separate run for every (LR, epochs) pair (much slower)
EXP2_LAYERS = [1, 3, 5, 7, 12]
EXP3_HEADS = [1, 2, 4, 8]
EXP4_ABLATIONS = {
    "Full model (baseline)": {},
    "No LayerNorm": {"use_norm": False},
    "No shortcut": {"use_shortcut": False},
    "No feed-forward": {"use_ffn": False},
}

# ---- Evaluation / text generation ----
SAMPLE_PROMPTS = ["Mr. Utterson the lawyer was", "It was a dark night, and"]
SAMPLE_TOKENS = 50
TOPK, TEMPERATURE = 25, 1.0
EVAL_TRAIN_MAX_BATCHES = 20     # train loss is measured on up to this many batches (no dropout)
COPY_NGRAM = 8                  # generated 8-grams found verbatim in the training text = memorisation

if QUICK_TEST:
    MAX_CHARS = 60_000
    BASE_TRAIN_CFG["num_epochs"] = 2
    EXP1_EPOCH_CHECKPOINTS = [1, 2, 3]
    EXP2_LAYERS = [1, 3]
    EXP3_HEADS = [1, 4]
    RESULTS_DIR_NAME += "_quicktest"
# =======================================================================

In [ ]:
if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")
    RESULTS_DIR = f"/content/drive/MyDrive/{RESULTS_DIR_NAME}"
else:
    RESULTS_DIR = RESULTS_DIR_NAME
RUNS_DIR = os.path.join(RESULTS_DIR, "runs")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
for d in (RUNS_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)
print("Results will be saved in:", RESULTS_DIR)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

## 1. Dataset: Project Gutenberg

In [ ]:
def fetch_gutenberg(book_id):
    cache = f"pg{book_id}.txt"
    if os.path.exists(cache):
        return open(cache, encoding="utf-8").read()
    urls = [f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
            f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt"]
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            raw = urllib.request.urlopen(req, timeout=30).read().decode("utf-8-sig", errors="ignore")
            open(cache, "w", encoding="utf-8").write(raw)
            return raw
        except Exception as e:
            print(f"  could not download {url}: {e}")
    raise RuntimeError(f"Book {book_id} not downloaded. Upload it manually and add it to LOCAL_BOOK_FILES.")

def clean_gutenberg(raw):
    title = re.search(r"Title:\s*(.+)", raw)
    title = title.group(1).strip() if title else "unknown"
    start = re.search(r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG E?BOOK.*?\*\*\*", raw, re.I | re.S)
    end = re.search(r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG E?BOOK", raw, re.I)
    body = raw[start.end() if start else 0: end.start() if end else len(raw)]
    body = body.replace("\r\n", "\n")
    body = re.sub(r"\n{3,}", "\n\n", body).strip()
    return title, body

books = []
for bid in BOOK_IDS:
    title, body = clean_gutenberg(fetch_gutenberg(bid))
    books.append({"id": bid, "title": title, "chars": len(body), "text": body})
for path in LOCAL_BOOK_FILES:
    title, body = clean_gutenberg(open(path, encoding="utf-8", errors="ignore").read())
    books.append({"id": path, "title": title if title != "unknown" else os.path.basename(path),
                  "chars": len(body), "text": body})
assert books, "No books loaded."

text_data = "\n\n<|endoftext|>\n\n".join(b["text"] for b in books)
if MAX_CHARS:
    text_data = text_data[:MAX_CHARS]

tokenizer = tiktoken.get_encoding("gpt2")
def encode(t): return tokenizer.encode(t, allowed_special={"<|endoftext|>"})

split_idx = int(TRAIN_RATIO * len(text_data))
train_text, val_text = text_data[:split_idx], text_data[split_idx:]
train_ids, val_ids = encode(train_text), encode(val_text)

for b in books:
    print(f"Book {b['id']}: {b['title']}  ({b['chars']:,} chars)")
print(f"Corpus: {len(text_data):,} chars | train {len(train_ids):,} tokens | val {len(val_ids):,} tokens")
assert len(val_ids) > BASE_MODEL_CFG["context_length"], "Validation split shorter than the context length: use a longer book."

In [ ]:
class TokenDataset(Dataset):
    # Same sliding-window input/target construction as GPTDatasetV1, on pre-tokenised ids.
    def __init__(self, token_ids, max_length, stride):
        self.inputs, self.targets = [], []
        for i in range(0, len(token_ids) - max_length, stride):
            self.inputs.append(torch.tensor(token_ids[i:i + max_length]))
            self.targets.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))
    def __len__(self): return len(self.inputs)
    def __getitem__(self, i): return self.inputs[i], self.targets[i]

def make_loaders(context_length, batch_size, seed):
    tr = TokenDataset(train_ids, context_length, context_length)
    va = TokenDataset(val_ids, context_length, context_length)
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(tr, batch_size=batch_size, shuffle=True, drop_last=True, generator=g)
    train_eval_loader = DataLoader(tr, batch_size=batch_size, shuffle=False, drop_last=False)
    val_loader = DataLoader(va, batch_size=batch_size, shuffle=False, drop_last=False)
    assert len(train_loader) > 0, "Too little training data for this batch size / context length."
    return train_loader, train_eval_loader, val_loader

_tl, _, _vl = make_loaders(BASE_MODEL_CFG["context_length"], BASE_TRAIN_CFG["batch_size"], SEED)
DATASET_STATS = {
    "books": [{k: v for k, v in b.items() if k != "text"} for b in books],
    "total_chars": len(text_data), "train_chars": len(train_text), "val_chars": len(val_text),
    "train_tokens": len(train_ids), "val_tokens": len(val_ids),
    "train_sequences": len(_tl.dataset), "val_sequences": len(_vl.dataset),
    "steps_per_epoch": len(_tl), "context_length": BASE_MODEL_CFG["context_length"],
    "batch_size": BASE_TRAIN_CFG["batch_size"], "tokenizer": "GPT-2 BPE (tiktoken), vocab 50257",
}
DATA_SIGNATURE = hashlib.md5(text_data.encode()).hexdigest()[:10]
json.dump(DATASET_STATS, open(os.path.join(RESULTS_DIR, "dataset_stats.json"), "w"), indent=2)
print(json.dumps(DATASET_STATS, indent=2))

# n-gram index of the training text, used to measure memorisation in generated samples
TRAIN_NGRAMS = {hash(tuple(train_ids[i:i + COPY_NGRAM])) for i in range(len(train_ids) - COPY_NGRAM + 1)}

## 2. Model: the class GPT with ablation switches

The code matches the class notebook. The only change is in `TransformerBlock` and `GPTModel`, where three config flags can turn components off:
- `use_norm=False` replaces `norm1` and `norm2` in every block with `nn.Identity`.
- `use_shortcut=False` drops the `x + shortcut` residual additions.
- `use_ffn=False` removes the whole feed-forward sub-layer, so each block has attention only.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, _ = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), GELU(),
                                    nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.use_shortcut = cfg.get("use_shortcut", True)
        self.use_ffn = cfg.get("use_ffn", True)
        self.att = MultiHeadAttention(d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
                                      context_length=cfg["context_length"], num_heads=cfg["n_heads"],
                                      dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])
        Norm = LayerNorm if cfg.get("use_norm", True) else nn.Identity
        self.norm1 = Norm(cfg["emb_dim"])
        self.ff = FeedForward(cfg) if self.use_ffn else None
        self.norm2 = Norm(cfg["emb_dim"]) if self.use_ffn else None
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x                       # attention sub-layer
        x = self.drop_shortcut(self.att(self.norm1(x)))
        if self.use_shortcut:
            x = x + shortcut
        if self.use_ffn:                   # feed-forward sub-layer
            shortcut = x
            x = self.drop_shortcut(self.ff(self.norm2(x)))
            if self.use_shortcut:
                x = x + shortcut
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"]) if cfg.get("use_final_norm", True) else nn.Identity()
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.trf_blocks(self.drop_emb(x))
        return self.out_head(self.final_norm(x))

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.tok_emb.weight.numel() + model.pos_emb.weight.numel() + model.out_head.weight.numel()
    return total, total - emb

_m = GPTModel(BASE_MODEL_CFG)
print("Base model parameters: total {:,} | non-embedding {:,}".format(*count_params(_m)))
del _m

## 3. Training and evaluation engine

In [ ]:
def calc_loss_batch(input_batch, target_batch, model):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return nn.functional.cross_entropy(logits.flatten(0, 1).float(), target_batch.flatten())

@torch.no_grad()
def calc_loss_loader(loader, model, max_batches=None):
    total, n = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
            total += calc_loss_batch(x, y, model).item()
        n += 1
    return total / max(n, 1)

def evaluate(model, train_eval_loader, val_loader):
    model.eval()
    tl = calc_loss_loader(train_eval_loader, model, EVAL_TRAIN_MAX_BATCHES)
    vl = calc_loss_loader(val_loader, model)
    model.train()
    return tl, vl

@torch.no_grad()
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, generator=None):
    # Same as the class `generate` (top-k + temperature), with a private RNG so that sampling
    # does not change the training random stream.
    for _ in range(max_new_tokens):
        logits = model(idx[:, -context_size:])[:, -1, :].float()
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top_logits[:, -1:], torch.full_like(logits, -torch.inf), logits)
        if temperature > 0.0:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1, generator=generator)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def text_metrics(tokens):
    # Simple inference-quality indicators for the generated continuation.
    n = COPY_NGRAM
    bigrams = list(zip(tokens, tokens[1:]))
    grams = [hash(tuple(tokens[i:i + n])) for i in range(len(tokens) - n + 1)]
    return {
        "distinct_1": len(set(tokens)) / max(len(tokens), 1),        # low = repetitive
        "distinct_2": len(set(bigrams)) / max(len(bigrams), 1),
        "copy_rate": sum(g in TRAIN_NGRAMS for g in grams) / max(len(grams), 1),  # high = memorised
    }

def sample_texts(model, context_size):
    model.eval()
    out = {}
    for p in SAMPLE_PROMPTS:
        ids = torch.tensor([encode(p)], device=device)
        gen = torch.Generator(device=device).manual_seed(SEED)
        greedy = generate(model, ids, SAMPLE_TOKENS, context_size)[0].tolist()
        topk = generate(model, ids, SAMPLE_TOKENS, context_size, TEMPERATURE, TOPK, gen)[0].tolist()
        k = ids.shape[1]
        out[p] = {"greedy": tokenizer.decode(greedy), "topk": tokenizer.decode(topk),
                  "greedy_metrics": text_metrics(greedy[k:]), "topk_metrics": text_metrics(topk[k:])}
    model.train()
    return out

def run_experiment(label, model_overrides=None, train_overrides=None, checkpoints=None, verbose=True):
    # Train one configuration and save everything to RUNS_DIR/<hash>.json.
    # Identical configurations are loaded from disk instead of retrained.
    mcfg = {**BASE_MODEL_CFG, **(model_overrides or {})}
    tcfg = {**BASE_TRAIN_CFG, **(train_overrides or {})}
    checkpoints = sorted(set(checkpoints or []))
    key = {"model": mcfg, "train": tcfg, "seed": SEED, "data": DATA_SIGNATURE, "amp": USE_AMP,
           "checkpoints": checkpoints, "prompts": SAMPLE_PROMPTS, "sample_tokens": SAMPLE_TOKENS}
    h = hashlib.md5(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12]
    path = os.path.join(RUNS_DIR, f"{h}.json")
    if os.path.exists(path):
        r = json.load(open(path))
        r["label"] = label
        print(f"[cached] {label}: best val {r['best_val']:.3f} @ epoch {r['best_val_epoch']}")
        return r

    set_seed(SEED)
    train_loader, train_eval_loader, val_loader = make_loaders(mcfg["context_length"], tcfg["batch_size"], SEED)
    model = GPTModel(mcfg).to(device)
    n_total, n_nonemb = count_params(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["weight_decay"])
    use_amp = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    clip = tcfg["grad_clip"] if tcfg["grad_clip"] else float("inf")

    tl0, vl0 = evaluate(model, train_eval_loader, val_loader)
    H = {"epochs": [0], "train_loss": [tl0], "val_loss": [vl0], "train_running_loss": [float("nan")],
         "grad_norm": [float("nan")], "tokens_seen": [0], "step_loss": []}
    ckpt_losses, ckpt_samples = {}, {}
    diverged, diverged_epoch, tokens_seen = False, None, 0
    print_every = max(1, tcfg["num_epochs"] // 15)
    t0 = time.time()

    for epoch in range(1, tcfg["num_epochs"] + 1):
        model.train()
        run_loss, gnorms, nb = 0.0, [], 0
        for x, y in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = calc_loss_batch(x, y, model)
            if not torch.isfinite(loss):
                diverged = True
                break
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
            lv = loss.item()
            H["step_loss"].append(lv)
            run_loss += lv; gnorms.append(gn.item()); nb += 1
            tokens_seen += x.numel()
        if diverged:
            diverged_epoch = epoch
            print(f"  !! {label}: loss became NaN/inf in epoch {epoch}, stopping this run.")
            break
        tl, vl = evaluate(model, train_eval_loader, val_loader)
        H["epochs"].append(epoch); H["train_loss"].append(tl); H["val_loss"].append(vl)
        H["train_running_loss"].append(run_loss / max(nb, 1))
        H["grad_norm"].append(float(np.nanmean(gnorms)) if gnorms else float("nan"))
        H["tokens_seen"].append(tokens_seen)
        if not (math.isfinite(tl) and math.isfinite(vl)):
            diverged, diverged_epoch = True, epoch
            print(f"  !! {label}: evaluation loss is NaN/inf at epoch {epoch}, stopping this run.")
            break
        if epoch in checkpoints:
            ckpt_losses[epoch] = {"train": tl, "val": vl}
            ckpt_samples[epoch] = sample_texts(model, mcfg["context_length"])
        if verbose and (epoch % print_every == 0 or epoch == 1 or epoch == tcfg["num_epochs"]):
            print(f"  [{label}] ep {epoch:4d}/{tcfg['num_epochs']}  train {tl:.3f}  val {vl:.3f}  "
                  f"|grad| {H['grad_norm'][-1]:.2f}  ({time.time() - t0:.0f}s)")

    elapsed = time.time() - t0
    val = np.array(H["val_loss"], dtype=float)
    best_i = int(np.nanargmin(val)) if np.isfinite(val).any() else 0
    final_samples = sample_texts(model, mcfg["context_length"]) if not diverged else {}
    epochs_run = H["epochs"][-1]
    r = {"label": label, "hash": h, "model_cfg": mcfg, "train_cfg": tcfg, "seed": SEED,
         "params_total": n_total, "params_non_embedding": n_nonemb, **H,
         "diverged": diverged, "diverged_epoch": diverged_epoch, "epochs_completed": epochs_run,
         "time_sec": elapsed, "sec_per_epoch": elapsed / max(epochs_run, 1),
         "best_val": float(val[best_i]), "best_val_epoch": H["epochs"][best_i],
         "final_train": H["train_loss"][-1], "final_val": H["val_loss"][-1],
         "checkpoint_losses": ckpt_losses, "checkpoint_samples": ckpt_samples,
         "final_samples": final_samples}
    json.dump(r, open(path, "w"))
    r = json.load(open(path))   # normalise (int keys -> str) so cached and fresh runs look the same
    r["label"] = label
    print(f"  done {label}: final train {r['final_train']:.3f} | final val {r['final_val']:.3f} | "
          f"best val {r['best_val']:.3f} @ ep {r['best_val_epoch']} | {elapsed/60:.1f} min")
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return r

In [ ]:
# ----------------------------- Reporting helpers -----------------------------
def ppl(x):
    return float(math.exp(x)) if x is not None and math.isfinite(x) and x < 50 else float("nan")

def summarize(results, extra_cols=None):
    rows = []
    for r in results:
        fs = r.get("final_samples") or {}
        gm = [v["greedy_metrics"] for v in fs.values()]
        row = {"Run": r["label"]}
        for name, fn in (extra_cols or {}).items():
            row[name] = fn(r)
        row.update({
            "Params (M)": r["params_total"] / 1e6,
            "Non-emb params (M)": r["params_non_embedding"] / 1e6,
            "Epochs": r["epochs_completed"],
            "Final train loss": r["final_train"],
            "Final val loss": r["final_val"],
            "Gap (val-train)": r["final_val"] - r["final_train"],
            "Best val loss": r["best_val"],
            "Best val epoch": r["best_val_epoch"],
            "Best val PPL": ppl(r["best_val"]),
            "Mean grad norm": float(np.nanmean(r["grad_norm"][1:])) if len(r["grad_norm"]) > 1 else float("nan"),
            "Sec/epoch": r["sec_per_epoch"],
            "Diverged": r["diverged"],
            "Distinct-2 (greedy)": float(np.mean([g["distinct_2"] for g in gm])) if gm else float("nan"),
            "Copy rate (greedy)": float(np.mean([g["copy_rate"] for g in gm])) if gm else float("nan"),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def save_table(df, name, caption=""):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False)
    try:
        tex = df.to_latex(index=False, float_format="%.3f", escape=True, caption=caption, label=f"tab:{name}")
        open(os.path.join(TAB_DIR, f"{name}.tex"), "w").write(tex)
    except Exception as e:
        print("LaTeX export skipped:", e)
    display(df.round(3))

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

def colors_for(n, ordinal=True):
    return plt.cm.viridis(np.linspace(0, 0.9, n)) if ordinal else plt.cm.tab10(np.arange(n))

def plot_overlay(results, title, name, ordinal=True, logx=False, ylim=None):
    # Two panels: training loss and validation loss of every run, one colour per run.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for r, c in zip(results, colors_for(len(results), ordinal)):
        ep = np.array(r["epochs"]); s = slice(1, None) if logx else slice(None)
        axes[0].plot(ep[s], np.array(r["train_loss"])[s], color=c, lw=1.6, label=r["label"])
        axes[1].plot(ep[s], np.array(r["val_loss"])[s], color=c, lw=1.6, label=r["label"])
        axes[1].scatter([r["best_val_epoch"]], [r["best_val"]], color=c, s=25, zorder=3)
    for ax, t in zip(axes, ["Training loss", "Validation loss (dot = best epoch)"]):
        ax.set_title(t); ax.set_xlabel("Epoch" + (" (log scale)" if logx else ""))
        if logx: ax.set_xscale("log")
        if ylim: ax.set_ylim(*ylim)
    axes[0].set_ylabel("Cross-entropy loss")
    axes[1].legend(fontsize=8, frameon=False)
    fig.suptitle(title, fontweight="bold"); fig.tight_layout()
    savefig(fig, name)

def plot_grid(results, title, name, ncols=3, logx=False, checkpoints=None):
    # One panel per run with train (solid) and validation (dashed) loss.
    n = len(results); ncols = min(ncols, n); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.4 * nrows), squeeze=False)
    for ax, r in zip(axes.flat, results):
        ep = np.array(r["epochs"]); s = slice(1, None) if logx else slice(None)
        ax.plot(ep[s], np.array(r["train_loss"])[s], label="Train", color="#1f77b4")
        ax.plot(ep[s], np.array(r["val_loss"])[s], "--", label="Validation", color="#d62728")
        ax.axvline(r["best_val_epoch"], color="gray", lw=0.8, ls=":")
        for c in (checkpoints or []):
            if c <= ep[-1]: ax.axvline(c, color="k", lw=0.4, alpha=0.25)
        t = r["label"] + (f"  [diverged ep {r['diverged_epoch']}]" if r["diverged"] else "")
        ax.set_title(t, fontsize=9); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        if logx: ax.set_xscale("log")
        ax.legend(fontsize=7, frameon=False)
    for ax in list(axes.flat)[n:]: ax.axis("off")
    fig.suptitle(title, fontweight="bold"); fig.tight_layout()
    savefig(fig, name)

def plot_bars(results, title, name):
    labels = [r["label"] for r in results]; x = np.arange(len(labels)); w = 0.27
    fig, ax = plt.subplots(figsize=(max(6, 1.5 * len(labels)), 3.8))
    for i, (key, lab) in enumerate([("final_train", "Final train"), ("final_val", "Final val"), ("best_val", "Best val")]):
        v = [r[key] for r in results]
        b = ax.bar(x + (i - 1) * w, v, w, label=lab)
        ax.bar_label(b, fmt="%.2f", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Cross-entropy loss"); ax.set_title(title, fontweight="bold"); ax.legend(fontsize=8, frameon=False)
    fig.tight_layout(); savefig(fig, name)

def plot_vs(results, xvals, xlabel, title, name):
    # Loss / speed / size as a function of the swept hyper-parameter.
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    axes[0].plot(xvals, [r["final_train"] for r in results], "o-", label="Final train")
    axes[0].plot(xvals, [r["final_val"] for r in results], "s--", label="Final val")
    axes[0].plot(xvals, [r["best_val"] for r in results], "^-", label="Best val")
    axes[0].set_ylabel("Loss"); axes[0].legend(fontsize=8, frameon=False)
    axes[1].plot(xvals, [r["sec_per_epoch"] for r in results], "o-", color="#9467bd"); axes[1].set_ylabel("Seconds / epoch")
    axes[2].plot(xvals, [r["params_non_embedding"] / 1e6 for r in results], "o-", color="#8c564b"); axes[2].set_ylabel("Non-embedding params (M)")
    for ax in axes: ax.set_xlabel(xlabel); ax.set_xticks(xvals)
    fig.suptitle(title, fontweight="bold"); fig.tight_layout(); savefig(fig, name)

def plot_gradnorm(results, title, name, ordinal=False):
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for r, c in zip(results, colors_for(len(results), ordinal)):
        ax.plot(r["epochs"][1:], r["grad_norm"][1:], color=c, label=r["label"])
    ax.set_yscale("log"); ax.set_xlabel("Epoch"); ax.set_ylabel("Mean gradient L2 norm (log)")
    ax.set_title(title, fontweight="bold"); ax.legend(fontsize=8, frameon=False)
    fig.tight_layout(); savefig(fig, name)

def print_samples(results, prompt_index=0):
    p = SAMPLE_PROMPTS[prompt_index]
    for r in results:
        s = (r.get("final_samples") or {}).get(p)
        print(f"--- {r['label']} ---")
        print("  greedy:", s["greedy"].replace("\n", " ") if s else "(diverged, no sample)")
        print("  top-k :", s["topk"].replace("\n", " ") if s else "")

### Time estimate
Runs a few training steps and estimates how long each experiment will take on this GPU. If the total is too long, set `USE_AMP = True`, lower `MAX_CHARS`, or drop 500 from `EXP1_EPOCH_CHECKPOINTS`.

In [ ]:
def time_per_step(model_overrides=None, steps=8):
    set_seed(SEED)
    cfg = {**BASE_MODEL_CFG, **(model_overrides or {})}
    tl, _, _ = make_loaders(cfg["context_length"], BASE_TRAIN_CFG["batch_size"], SEED)
    m = GPTModel(cfg).to(device); opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    it = iter(tl); x, y = next(it)
    for i in range(steps + 2):
        if i == 2:
            if device.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
        opt.zero_grad(); calc_loss_batch(x, y, m).backward(); opt.step()
    if device.type == "cuda": torch.cuda.synchronize()
    dt = (time.time() - t0) / steps
    del m, opt
    return dt

spe = DATASET_STATS["steps_per_epoch"]; overhead = 1.35   # evaluation + sampling overhead
base_step = time_per_step()
deep_step = time_per_step({"n_layers": max(EXP2_LAYERS)})
per_layer = (deep_step - base_step) / (max(EXP2_LAYERS) - BASE_MODEL_CFG["n_layers"])
est_step = lambda L: max(base_step + per_layer * (L - BASE_MODEL_CFG["n_layers"]), 1e-4)
E = BASE_TRAIN_CFG["num_epochs"]
if EXP1_INDEPENDENT_RUNS:
    exp1_epochs = len(EXP1_LRS) * sum(EXP1_EPOCH_CHECKPOINTS)
else:
    exp1_epochs = len(EXP1_LRS) * max(EXP1_EPOCH_CHECKPOINTS)
est = {
    "Baseline": E * spe * base_step,
    "Exp 1 (epochs x LR)": exp1_epochs * spe * base_step,
    "Exp 2 (layers)": sum(E * spe * est_step(L) for L in EXP2_LAYERS),
    "Exp 3 (heads)": len(EXP3_HEADS) * E * spe * base_step,
    "Exp 4 (ablations)": (len(EXP4_ABLATIONS) - 1) * E * spe * base_step,
}
print(f"{spe} steps/epoch, {base_step*1000:.0f} ms/step (base model)\n")
for k, v in est.items():
    print(f"{k:22s} ~ {v * overhead / 60:6.1f} min")
print(f"{'TOTAL':22s} ~ {sum(est.values()) * overhead / 60:6.1f} min")

## 4. Baseline training
Base model (6 layers, 4 heads, 256-dim embeddings, context 256), AdamW with lr 4e-4 and weight decay 0.1, batch size 8, 30 epochs.

In [ ]:
baseline = run_experiment("Baseline")
plot_grid([baseline], "Baseline: training vs validation loss", "baseline_loss", ncols=1)
df_base = summarize([baseline])
save_table(df_base, "baseline_summary", "Baseline training results.")
print_samples([baseline], 0); print_samples([baseline], 1)

## 5. Experiment 1: Epochs × learning rate
The learning rate is held constant, so the first *E* epochs of a 500-epoch run are exactly an *E*-epoch run (same seed and data order). Each LR is therefore trained once for `max(EXP1_EPOCH_CHECKPOINTS)` epochs, and the losses and generated text are recorded at every checkpoint (5, 10, 20, 50, 100, 200, 500).

**What to look for:** how speed of convergence depends on LR; where validation loss bottoms out (the best number of epochs) and then rises (overfitting); whether 1e-2 is unstable or diverges; the growing train/val gap; and copy rate going up as the model memorises the book.

In [ ]:
exp1 = []
lr_name = lambda lr: f"{lr:.0e}".replace("e-0", "e-")
if not EXP1_INDEPENDENT_RUNS:
    for lr in EXP1_LRS:
        exp1.append(run_experiment(f"LR {lr_name(lr)}", train_overrides={"lr": lr, "num_epochs": max(EXP1_EPOCH_CHECKPOINTS)},
                                   checkpoints=EXP1_EPOCH_CHECKPOINTS))
else:
    exp1_indep = {}
    for lr in EXP1_LRS:
        for E in EXP1_EPOCH_CHECKPOINTS:
            exp1_indep[(lr, E)] = run_experiment(f"LR {lr_name(lr)}, {E} ep", train_overrides={"lr": lr, "num_epochs": E}, checkpoints=[E])
        exp1.append(exp1_indep[(lr, max(EXP1_EPOCH_CHECKPOINTS))])
        exp1[-1]["label"] = f"LR {lr_name(lr)}"

In [ ]:
plot_grid(exp1, "Exp 1: train/validation loss per learning rate (vertical lines = epoch checkpoints)",
          "exp1_grid_linear", ncols=3, checkpoints=EXP1_EPOCH_CHECKPOINTS)
plot_grid(exp1, "Exp 1: same, log-scale epochs", "exp1_grid_logx", ncols=3, logx=True, checkpoints=EXP1_EPOCH_CHECKPOINTS)
plot_overlay(exp1, "Exp 1: learning-rate comparison", "exp1_overlay", ordinal=True, logx=True)

# Checkpoint table: loss at every (LR, epochs) combination
rows = []
for lr, r in zip(EXP1_LRS, exp1):
    for E in EXP1_EPOCH_CHECKPOINTS:
        if EXP1_INDEPENDENT_RUNS:
            rr = exp1_indep[(lr, E)]; cl = rr["checkpoint_losses"].get(str(E)); cs = rr["checkpoint_samples"].get(str(E), {})
        else:
            cl = r["checkpoint_losses"].get(str(E)); cs = r["checkpoint_samples"].get(str(E), {})
        gm = [v["greedy_metrics"] for v in cs.values()]
        rows.append({"LR": lr_name(lr), "Epochs": E,
                     "Train loss": cl["train"] if cl else float("nan"),
                     "Val loss": cl["val"] if cl else float("nan"),
                     "Gap": (cl["val"] - cl["train"]) if cl else float("nan"),
                     "Val PPL": ppl(cl["val"]) if cl else float("nan"),
                     "Distinct-2": float(np.mean([g["distinct_2"] for g in gm])) if gm else float("nan"),
                     "Copy rate": float(np.mean([g["copy_rate"] for g in gm])) if gm else float("nan"),
                     "Status": "ok" if cl else "diverged / not reached"})
df_exp1_ckpt = pd.DataFrame(rows)
save_table(df_exp1_ckpt, "exp1_checkpoints", "Loss at each epoch checkpoint for each learning rate.")

# Heatmaps (LR x epochs)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
for ax, col in zip(axes, ["Train loss", "Val loss"]):
    M = df_exp1_ckpt.pivot(index="LR", columns="Epochs", values=col).reindex([lr_name(l) for l in EXP1_LRS])
    im = ax.imshow(M.values, cmap="viridis_r", aspect="auto")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns); ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(M.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            ax.text(j, i, "div" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if (not np.isnan(v) and v > np.nanmean(M.values)) else "black")
    ax.set_xlabel("Epochs"); ax.set_ylabel("Learning rate"); ax.set_title(col); ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.suptitle("Exp 1: loss at each (learning rate, epochs) combination", fontweight="bold"); fig.tight_layout()
savefig(fig, "exp1_heatmap")

df_exp1 = summarize(exp1, {"LR": lambda r: r["train_cfg"]["lr"]})
save_table(df_exp1, "exp1_summary", "Experiment 1 summary per learning rate.")
plot_gradnorm(exp1, "Exp 1: gradient norm during training", "exp1_gradnorm", ordinal=True)

In [ ]:
# How the generated text changes with training length (greedy decoding, prompt 1)
p = SAMPLE_PROMPTS[0]
for lr, r in zip(EXP1_LRS, exp1):
    print(f"\n================ LR {lr_name(lr)} ================")
    for E in EXP1_EPOCH_CHECKPOINTS:
        src = exp1_indep[(lr, E)] if EXP1_INDEPENDENT_RUNS else r
        s = src["checkpoint_samples"].get(str(E), {}).get(p)
        print(f"[{E:>3} ep] " + (s["greedy"].replace("\n", " ") if s else "(diverged / not reached)"))

## 6. Experiment 2: Number of transformer layers (1, 3, 5, 7, 12)
Everything else is the baseline setup (4 heads, 256 dims, lr 4e-4, same number of epochs).

**What to look for:** whether deeper models reach lower training loss (more capacity), whether validation improves or overfits sooner on a small book, and the cost (time per epoch, parameters) of extra depth.

In [ ]:
exp2 = [run_experiment(f"{L} layer" + ("s" if L > 1 else ""), model_overrides={"n_layers": L}) for L in EXP2_LAYERS]
plot_overlay(exp2, "Exp 2: effect of the number of transformer layers", "exp2_overlay", ordinal=True)
plot_grid(exp2, "Exp 2: train/validation loss per depth", "exp2_grid", ncols=3)
plot_vs(exp2, EXP2_LAYERS, "Number of transformer layers", "Exp 2: loss, speed and size vs depth", "exp2_vs_layers")
plot_gradnorm(exp2, "Exp 2: gradient norm vs depth", "exp2_gradnorm", ordinal=True)
df_exp2 = summarize(exp2, {"Layers": lambda r: r["model_cfg"]["n_layers"]})
save_table(df_exp2, "exp2_summary", "Experiment 2: varying the number of transformer layers.")
print_samples(exp2)

## 7. Experiment 3: Number of attention heads (1, 2, 4, 8)
The embedding size stays at 256, so the size of each head is 256/h. The parameter count is **identical** for every head count, so only how attention is split changes.

**What to look for:** whether several smaller heads (which can attend to different positions or relations at once) beat one big head, and whether very small heads (32 dims at h=8) start to hurt.

In [ ]:
exp3 = [run_experiment(f"{h} head" + ("s" if h > 1 else ""), model_overrides={"n_heads": h}) for h in EXP3_HEADS]
plot_overlay(exp3, "Exp 3: effect of the number of attention heads", "exp3_overlay", ordinal=True)
plot_grid(exp3, "Exp 3: train/validation loss per head count", "exp3_grid", ncols=4)
plot_vs(exp3, EXP3_HEADS, "Number of attention heads", "Exp 3: loss, speed and size vs heads", "exp3_vs_heads")
df_exp3 = summarize(exp3, {"Heads": lambda r: r["model_cfg"]["n_heads"],
                           "Head dim": lambda r: r["model_cfg"]["emb_dim"] // r["model_cfg"]["n_heads"]})
save_table(df_exp3, "exp3_summary", "Experiment 3: varying the number of attention heads.")
print_samples(exp3)

## 8. Experiment 4: Ablation studies
Each run removes one component from **every** transformer block. The final LayerNorm before the output head is kept; set `use_final_norm: False` to remove it too.

**What to look for:**
- **No LayerNorm:** larger and less stable activations and gradient norms, slower or noisier optimisation.
- **No shortcut:** gradients have to pass through every sub-layer, so deep stacks train much worse (watch the gradient norm and how high the loss plateaus).
- **No feed-forward:** about two-thirds fewer non-embedding parameters and no per-token non-linear processing, so the model memorises less and its text is less fluent.

Judge inference quality with the samples and the distinct-2 and copy-rate columns.

In [ ]:
exp4 = [run_experiment(name, model_overrides=ov) for name, ov in EXP4_ABLATIONS.items()]
plot_overlay(exp4, "Exp 4: ablation of transformer components", "exp4_overlay", ordinal=False)
plot_grid(exp4, "Exp 4: train/validation loss per ablation", "exp4_grid", ncols=4)
plot_bars(exp4, "Exp 4: final and best losses per ablation", "exp4_bars")
plot_gradnorm(exp4, "Exp 4: gradient norm per ablation", "exp4_gradnorm", ordinal=False)

# Per-step training loss (shows instability better than per-epoch averages)
fig, ax = plt.subplots(figsize=(8, 3.8))
for r, c in zip(exp4, colors_for(len(exp4), False)):
    sl = pd.Series(r["step_loss"]).rolling(10, min_periods=1).mean()
    ax.plot(np.arange(len(sl)) / DATASET_STATS["steps_per_epoch"], sl, color=c, label=r["label"], lw=1.2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Train batch loss (10-step moving avg)")
ax.set_title("Exp 4: step-level training loss", fontweight="bold"); ax.legend(fontsize=8, frameon=False)
fig.tight_layout(); savefig(fig, "exp4_step_loss")

df_exp4 = summarize(exp4)
save_table(df_exp4, "exp4_summary", "Experiment 4: ablation study.")
print_samples(exp4, 0); print_samples(exp4, 1)

## 9. Export everything for the report

In [ ]:
# Master summary across all runs
all_runs = [("Baseline", baseline)] + [("Exp1", r) for r in exp1] + [("Exp2", r) for r in exp2] \
         + [("Exp3", r) for r in exp3] + [("Exp4", r) for r in exp4]
df_all = summarize([r for _, r in all_runs])
df_all.insert(0, "Experiment", [e for e, _ in all_runs])
save_table(df_all, "summary_all", "Summary of all training runs.")

# All generated text in one Markdown file
with open(os.path.join(RESULTS_DIR, "samples.md"), "w") as f:
    f.write(f"# Generated samples\n\nGreedy decoding and top-k={TOPK}, T={TEMPERATURE} sampling, {SAMPLE_TOKENS} new tokens.\n")
    for exp, r in all_runs:
        f.write(f"\n## {exp}: {r['label']}\n")
        for p, s in (r.get("final_samples") or {}).items():
            f.write(f"\n**Prompt:** `{p}`\n\n- greedy: {s['greedy']!r}\n- top-k: {s['topk']!r}\n"
                    f"- greedy metrics: {json.dumps({k: round(v, 3) for k, v in s['greedy_metrics'].items()})}\n")
        for E, ss in sorted(r.get("checkpoint_samples", {}).items(), key=lambda kv: int(kv[0])):
            for p, s in ss.items():
                f.write(f"\n*Epoch {E}*, `{p}`: {s['greedy']!r}\n")

# Environment + configuration, for the report's setup section
env = {"gpu": GPU_NAME, "torch": torch.__version__, "python": platform.python_version(),
       "seed": SEED, "use_amp": USE_AMP, "base_model_cfg": BASE_MODEL_CFG, "base_train_cfg": BASE_TRAIN_CFG,
       "exp1_lrs": EXP1_LRS, "exp1_checkpoints": EXP1_EPOCH_CHECKPOINTS, "exp1_independent_runs": EXP1_INDEPENDENT_RUNS,
       "exp2_layers": EXP2_LAYERS, "exp3_heads": EXP3_HEADS, "exp4_ablations": EXP4_ABLATIONS,
       "sample_prompts": SAMPLE_PROMPTS, "copy_ngram": COPY_NGRAM, "quick_test": QUICK_TEST}
json.dump(env, open(os.path.join(RESULTS_DIR, "config_and_environment.json"), "w"), indent=2)

# Index of runs: label -> json file
json.dump({f"{e}: {r['label']}": f"runs/{r['hash']}.json" for e, r in all_runs},
          open(os.path.join(RESULTS_DIR, "runs_index.json"), "w"), indent=2)

# Zip everything (figures, tables, run logs, samples)
zip_path = f"{RESULTS_DIR_NAME}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk(RESULTS_DIR):
        for fn in fnames:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, os.path.dirname(RESULTS_DIR.rstrip("/")) or "."))
print("Saved", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Figures:", sorted(os.listdir(FIG_DIR)))
print("Tables :", sorted(os.listdir(TAB_DIR)))
if IN_COLAB:
    colab_files.download(zip_path)